# Lesson 2 · Phase 3 — Computing Gradients, Backprop & Failure Modes

**Mastering Agentic AI Certification · Pre-read**

> A fresh network predicts nonsense — its weights are random (Phase 2). **Learning** means adjusting those weights to reduce error. This phase covers how we measure error, **compute gradients**, push them backward with **backpropagation**, and the two classic ways it breaks: **vanishing** and **exploding** gradients.

## 1. The complete picture (the training loop)

```
   FORWARD  ─────────────────────────────▶
 x ─▶ layers ─▶ prediction ─▶ LOSS (how wrong?)
 ^                                  |
 |                                  v  compute gradients ∂L/∂w
 update w := w - η·∂L/∂w   ◀──── BACKPROP (chain rule, layer by layer)
   ◀───────────────────────────── BACKWARD
```

Repeat over trillions of tokens and the weights drift toward values that make good predictions. **This loop is literally what "pre-training" and "fine-tuning" run.**

## 2. The loss — measuring "how wrong"

Training needs a single number to minimise. For next-token prediction that's **cross-entropy** (low when the model put high probability on the correct token):

$$\mathcal{L} = -\log P_\theta(\text{correct token})$$

If the right answer got 90% probability, loss ≈ 0.105 (small). If it got 1%, loss ≈ 4.6 (large). Learning = **make $\mathcal{L}$ smaller**.

## 3. Computing gradients — which way is "downhill"?

A **gradient** $\dfrac{\partial \mathcal{L}}{\partial w}$ answers: *if I nudge weight $w$ a little, how much does the loss change, and in which direction?*

- Positive gradient ⇒ increasing $w$ increases loss ⇒ we should **decrease** $w$.
- We step **opposite** the gradient — **gradient descent**:

$$w \leftarrow w - \eta \,\frac{\partial \mathcal{L}}{\partial w}$$

where $\eta$ is the **learning rate** (step size). The gradient is the local "downhill" direction on the loss surface.

## 4. Backpropagation — the chain rule, layer by layer

A deep net is a **composition** of functions. To get the gradient for an early weight we apply the **chain rule**, multiplying the local derivatives of each layer from the output back to the input:

$$\frac{\partial \mathcal{L}}{\partial w_{\text{early}}} = \frac{\partial \mathcal{L}}{\partial a_n}\cdot\frac{\partial a_n}{\partial a_{n-1}}\cdots\frac{\partial a_1}{\partial w_{\text{early}}}$$

**Backprop** = compute these products efficiently in **one backward sweep**, reusing results. The gradient *flows backward* through the same layers the data flowed forward through.

## 5. Differentiable ops are *required*

Backprop multiplies **derivatives**. So **every operation in the network must be differentiable** (have a usable slope) — otherwise the gradient can't pass through it. This single requirement explains many design choices:

| We use… | …instead of | Because |
|---|---|---|
| **softmax** | `argmax` (pick the top) | `argmax` is flat/jumpy → derivative 0 or undefined ⇒ no gradient |
| **smooth/ReLU activations** | hard step functions | step functions have 0 slope almost everywhere |
| **cross-entropy** | "count correct" accuracy | accuracy isn't differentiable |

> Rule of thumb: if you can't take its derivative, you can't backprop through it — so it can't be *learned* end-to-end.

In [ ]:
import numpy as np
# Concrete gradient via the chain rule on a 2-step net:  L = (relu(w2 * relu(w1 * x)) - y)^2
x, y, w1, w2 = 2.0, 1.0, 0.5, 0.8
# forward
h_pre = w1 * x;        h = max(0.0, h_pre)
o_pre = w2 * h;        o = max(0.0, o_pre)
L = (o - y) ** 2
# backward (chain rule), d/dz relu = 1 if z>0 else 0
dL_do  = 2 * (o - y)
do_dop = 1.0 if o_pre > 0 else 0.0
dL_dw2 = dL_do * do_dop * h                       # ∂L/∂w2
dL_dh  = dL_do * do_dop * w2
dh_dhp = 1.0 if h_pre > 0 else 0.0
dL_dw1 = dL_dh * dh_dhp * x                        # ∂L/∂w1 (earlier weight)
print(f"loss = {L:.4f}")
print(f"∂L/∂w2 = {dL_dw2:.4f}   ∂L/∂w1 = {dL_dw1:.4f}")
eta = 0.1
print(f"updated w1 = {w1 - eta*dL_dw1:.4f}, w2 = {w2 - eta*dL_dw2:.4f}  (one gradient-descent step)")

## 6. Failure mode A — Vanishing gradients

Backprop **multiplies** one derivative per layer. If each is **< 1** (e.g. sigmoid/tanh, whose slope is ≤ 0.25 and ≈ 0 when saturated), the product **shrinks exponentially with depth**:

$$0.25^{20} \approx 9\times10^{-13} \approx 0$$

Early layers then get a near-zero gradient ⇒ they **barely learn**; the deep network trains painfully slowly or stalls.

**Fixes:** ReLU/GELU (slope 1 for active units), **residual/skip connections** (a gradient "highway"), **normalization** (LayerNorm/BatchNorm), careful initialization. *Transformers rely heavily on residuals + LayerNorm precisely to keep gradients alive across dozens of layers.*

## 7. Failure mode B — Exploding gradients

The opposite: if per-layer derivatives are **> 1**, the product **blows up exponentially** with depth:

$$1.5^{20} \approx 3327, \qquad 2^{40} \approx 10^{12}$$

Gradients become huge ⇒ giant weight updates ⇒ the loss diverges to **NaN/Inf** and training destabilises.

**Fixes:** **gradient clipping** (cap the gradient norm), careful weight initialization (Xavier/He), normalization, lower learning rate.

In [ ]:
# COMPLETE PICTURE: gradient magnitude vs network depth — vanish vs explode vs healthy.
import numpy as np, matplotlib.pyplot as plt
depths = np.arange(1, 41)
vanish  = 0.6 ** depths     # per-layer factor < 1  (e.g. saturating sigmoid)
explode = 1.3 ** depths     # per-layer factor > 1
healthy = 1.0 ** depths     # per-layer factor ~ 1  (ReLU + residuals + norm)

plt.figure(figsize=(7,4))
plt.semilogy(depths, vanish,  "o-", label="factor 0.6  → VANISHING", ms=3)
plt.semilogy(depths, explode, "s-", label="factor 1.3  → EXPLODING", ms=3)
plt.semilogy(depths, healthy, "^-", label="factor 1.0  → healthy (ReLU+resid+norm)", ms=3)
plt.axhline(1, color="grey", lw=0.6)
plt.xlabel("layer depth (backprop steps)"); plt.ylabel("gradient magnitude (log scale)")
plt.title("Why depth breaks naive backprop"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("at depth 40:  vanishing →", f"{0.6**40:.2e}", " exploding →", f"{1.3**40:.2e}")

## 8. How this contributes — and the link back to the Transformer

This loop is **how every weight in Lessons 1–3 actually gets its value**:

- **Pre-training, fine-tuning, and RLHF/DPO** are all *the same gradient-descent loop* — forward pass → loss → backprop → update — differing only in the data and the loss.
- Because backprop needs **differentiable ops**, the Transformer is built entirely from differentiable pieces (embeddings, linear layers, softmax attention, smooth activations) — no `argmax` inside the path that learns.
- The Transformer's signature components — **residual connections** and **LayerNorm** — exist largely to **defeat vanishing/exploding gradients**, letting gradients flow cleanly through dozens of stacked layers.

```
 Lesson 2 mechanics  ──powers──▶  Lesson 1 pre-training, Lesson(FT) fine-tuning, Lesson(Align) alignment
 (embeddings · forward pass · gradients · backprop)        (all = this same training loop at scale)
```

## 9. Key takeaways

1. **Loss** (cross-entropy) measures error; training **minimises** it.
2. A **gradient** $\partial\mathcal{L}/\partial w$ is the downhill direction; **gradient descent** steps opposite it by the learning rate.
3. **Backprop** = the **chain rule** applied layer-by-layer in one backward sweep.
4. Every op must be **differentiable** — that's why softmax/cross-entropy replace argmax/accuracy.
5. **Vanishing** (factors <1 shrink to 0) and **exploding** (factors >1 blow up) gradients are the depth failure modes; **ReLU, residuals, normalization, and gradient clipping** are the fixes the Transformer relies on.

---
*This completes Lesson 2.* You now have the full picture: **text → embeddings → forward pass → prediction → loss → backprop → weight updates** — the machinery underneath every model in this course.

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0]); print("Platform:", platform.platform())
print("Lesson 2 · Phase 3 (Gradients, Backprop & Failure Modes) notebook is running ✓")